# Visual retrieval avec CLIP

Recherche texte-image et image-image sur une petite collection de trois photographies réelles. CLIP préentraîné ViT-B/32, aucun entraînement, pas de génération d’image. Premier téléchargement des poids nécessaire ; CPU possible, GPU facultatif.

**Statut : code préparé et vérifié syntaxiquement, inférence non exécutée dans cet environnement.** Exécuter de haut en bas dans Colab ou Jupyter. Les sorties sont volontairement vides, aucun résultat inventé. Petits tests sans entraînement. Python 3.10–3.12 recommandé.

## 1. Dépendances

In [ ]:
%pip install "torch>=2.2,<3" "transformers>=4.45,<5" "pillow>=10,<13" "matplotlib>=3.8,<4"

In [ ]:
from pathlib import Path
import time, urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
DATA=Path('data'); DATA.mkdir(exist_ok=True)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
def download(url, name):
    path=DATA/name
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return path


## 2. Petite collection réelle
Images publiques de démonstration : chien PyTorch, bus et football Ultralytics. Les droits restent ceux des sources. Remplacer par vos propres images pour un projet réel.

In [ ]:
sources=[
('dog','https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'),
('bus','https://ultralytics.com/images/bus.jpg'),
('football','https://ultralytics.com/images/zidane.jpg')]
images=[Image.open(download(url,name+'.jpg')).convert('RGB') for name,url in sources]
labels=[name for name,_ in sources]
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,im,label in zip(axes,images,labels): ax.imshow(im);ax.set_title(label);ax.axis('off')
plt.show()


## 3. Encodeurs CLIP et embeddings
Les représentations sont normalisées L2. La similarité cosinus n’est pas une probabilité. Les requêtes anglaises facilitent ce petit test du modèle.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
model_id='openai/clip-vit-base-patch32'
device='cuda' if torch.cuda.is_available() else 'cpu'
model=CLIPModel.from_pretrained(model_id).to(device).eval()
processor=CLIPProcessor.from_pretrained(model_id)
queries=['a photo of a dog','a photo of a bus','people playing football']
inputs=processor(text=queries,images=images,return_tensors='pt',padding=True).to(device)
if device=='cuda':torch.cuda.synchronize()
t0=time.perf_counter()
with torch.inference_mode():
    out=model(**inputs)
    I=out.image_embeds; I=I/I.norm(dim=-1,keepdim=True)
    T=out.text_embeds; T=T/T.norm(dim=-1,keepdim=True)
    similarities=(T@I.T).cpu().numpy()
if device=='cuda':torch.cuda.synchronize()
print('Appareil :',device, '| temps encodeurs :',round(time.perf_counter()-t0,3),'s')
print('Matrice texte x images :\n',similarities.round(3))
assert similarities.shape==(3,3) and np.isfinite(similarities).all()


## 4. Top-k et recherche image-image
Pour image-image, exclure l’image requête : sinon elle se retrouve elle-même. Trois images suffisent pour la mécanique, pas pour évaluer robustesse et généralisation.

In [ ]:
rank=np.argsort(-similarities,axis=1)
q=0
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,j in zip(axes,rank[q]):
    ax.imshow(images[j]);ax.set_title(f'{labels[j]} | cos={similarities[q,j]:.3f}');ax.axis('off')
plt.suptitle(queries[q]);plt.show()
image_scores=(I@I.T).detach().cpu().numpy()
np.fill_diagonal(image_scores,-np.inf)
print('Voisin du chien hors lui-même :',labels[int(image_scores[0].argmax())])
np.save(OUT/'image_embeddings.npy',I.detach().cpu().numpy())


## 5. Petit contrôle de pertinence
Une seule cible pertinente par requête, indices 0,1,2. Hit@k (aussi appelé Recall@k dans ce protocole à cible unique). Ici mAP est la moyenne de 1/rang de la cible. Les résultats ne sont pas un benchmark représentatif.

In [ ]:
targets=np.arange(3)
positions=np.array([int(np.flatnonzero(rank[i]==targets[i])[0])+1 for i in range(3)])
for k in [1,2,3]:print(f'Hit@{k} : {(positions<=k).mean():.3f}')
print('mAP, une cible par requête :',float(np.mean(1/positions)))
assert np.all((positions>=1)&(positions<=3))


## Questions de compréhension
1. Retrieval vs génération ?
2. Pourquoi normaliser les embeddings ?
3. Pourquoi retirer l’image requête en image-image ?
4. Pourquoi trois images ne constituent-elles pas un benchmark ?

## Bibliographie
- https://huggingface.co/openai/clip-vit-base-patch32
- https://arxiv.org/abs/2103.00020
- https://huggingface.co/docs/transformers/model_doc/clip